In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("Cluster_Process_Articles_Multimodal") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

data = ["hello world", "hello spark", "spark is great", "hello world spark"]
rdd = spark.sparkContext.parallelize(data)

counts = rdd.flatMap(lambda line: line.split(" ")) \
             .map(lambda word: (word, 1)) \
             .reduceByKey(lambda a, b: a + b)

df = counts.toDF(["word", "count"])
df.coalesce(1).write.mode("overwrite").csv("hdfs://namenode:9000/output/wordcount", header=True)

for word, count in counts.collect():
    print(f"{word}: {count}")

spark.stop()

26/04/02 15:05:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


world: 2
is: 1
hello: 3
spark: 3
great: 1
